In [1]:
# Create product catalog and price data with related but differently named keys
import pandas as pd
products = pd.DataFrame({
    "sku": ["A", "B", "C", "D"],
    "product_name": ["Pen", "Bag", "Book", "Bottle"],
    "category": ["stationery", "travel", "stationery", "travel"],
})

prices = pd.DataFrame({
    "product_code": ["A", "B", "C", "E"],
    "price": [2.5, 25.0, 8.0, 12.0],
})

In [2]:
# Left join keeps every product and adds matching price data
products_with_prices=products.merge(
    prices,
    left_on="sku",
    right_on="product_code",
    how="left"
)
products_with_prices

,sku,product_name,category,product_code,price
0,A,Pen,stationery,A,2.5
1,B,Bag,travel,B,25.0
2,C,Book,stationery,C,8.0
3,D,Bottle,travel,NaN,NaN


In [4]:
# Find catalog products that do not have a matching price
products_with_prices[products_with_prices["price"].isna()]

,sku,product_name,category,product_code,price
3,D,Bottle,travel,NaN,NaN


In [5]:
# Right join keeps every price row and adds matching catalog data
all_products_prices=products.merge(
    prices,
    left_on="sku",
    right_on="product_code",
    how="right"
)
print(all_products_prices)

   sku product_name    category product_code  price
0    A          Pen  stationery            A    2.5
1    B          Bag      travel            B   25.0
2    C         Book  stationery            C    8.0
3  NaN          NaN         NaN            E   12.0


In [7]:
# Find price records whose SKU is not present in the catalog
print("Prices that do not have a matching product:\n")
all_products_prices[all_products_prices["sku"].isna()]

Price that not having a matching product:



,sku,product_name,category,product_code,price
3,NaN,NaN,NaN,E,12.0


In [8]:
# Outer join keeps unmatched rows from both tables and marks their source
merged=products.merge(
    prices,
    left_on="sku",
    right_on="product_code",
    how="outer",
    indicator=True
)
merged

,sku,product_name,category,product_code,price,_merge
0,A,Pen,stationery,A,2.5,both
1,B,Bag,travel,B,25.0,both
2,C,Book,stationery,C,8.0,both
3,D,Bottle,travel,NaN,NaN,left_only
4,NaN,NaN,NaN,E,12.0,right_only


In [10]:
# Keep only records matched in both tables for analysis
analysis_ready_table=merged[merged["_merge"]=="both"]
analysis_ready_table

,sku,product_name,category,product_code,price,_merge
0,A,Pen,stationery,A,2.5,both
1,B,Bag,travel,B,25.0,both
2,C,Book,stationery,C,8.0,both


### Important notes

- Use `left_on` and `right_on` when the two tables use different key-column names.
- `how="left"` preserves all left rows, `how="right"` preserves all right rows, and `how="outer"` preserves rows from both sides.
- Missing values after a join can indicate an unmatched key, not necessarily a true missing price. Investigate unmatched rows before filling values.
- `indicator=True` adds `_merge` with `left_only`, `right_only`, or `both` labels for join diagnostics.
- Drop the redundant key or rename join keys when clean output is required, and verify row counts because joins can multiply rows if keys are duplicated.
